In [1]:
%pip install -quiet PyPDF2 opensearch-py sentence-transformers

Note: you may need to restart the kernel to use updated packages.



Usage:   
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <requirement specifier> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] -r <requirements file> [package-index-options] ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <vcs project url> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] [-e] <local project path> ...
  c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Scripts\python.exe 
-m pip install [options] <archive url/path> ...

no such option: -u


In [2]:
import json
import re
import os
import sys
from typing import Dict, List, Any

from PyPDF2 import PdfReader
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer

c:\Users\ksuma\OneDrive\Documents\local-rag-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
OPENSEARCH_HOST = "localhost"
OPENSEARCH_PORT = 9200
OPENSEARCH_INDEX ="documents"

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384
ASYMETRIC_EMBEDDING = False

TEXT_CHUNK_SIZE = 500
TEXT_CHUNK_OVERLAP = 100


In [4]:
def clean_text(text: str) -> str:
    """Cleans OCR-extracted text by removing unnecessary newlines, hyphens, and correcting common OCR errors.

    Args:
        text (str): The text to clean.

    Returns:
        str: The cleaned text."""

    text = re.sub(r"(\w+)-\n(\w+)", r"\1\2", text)
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)

    return text.strip()

In [5]:
def chunk_text(text: str, chunk_size: int, overlap: int = 100) -> List[str]:
    """Splits text into chunks with a specified overlap.

    Args:
        text (str): The text to split.
        chunk_size (int): The number of tokens in each chunk.
        overlap (int): The number of tokens to overlap between chunks.

    Returns:
        List[str]: A list of text chunks."""

    text = clean_text(text)
    tokens = text.split(" ")
    chunks = []
    start = 0
    while start <= len(tokens):
        end = start + chunk_size
        chunk_token = tokens[start:end]
        chunk_text = " ".join(chunk_token)
        chunks.append(chunk_text)
        start = end - overlap
    
    return chunks

In [6]:
def get_embedding_model():
    """Loads and returns the sentence transformer embedding model.
    
    Returns:
        SentenceTransformer: The loaded embedding model."""
    model = SentenceTransformer(EMBEDDING_MODEL)
    return model

def generate_embedding(texts: List[str]):
    """
    Generates embeddings for a list of text chunks.
    
    Args:
        texts (List[str]): List of text chunks to embed.
        
    Returns:
        List[numpy.ndarray]: List of embedding vectors.
    """
    model = get_embedding_model()

    if ASYMETRIC_EMBEDDING :
        texts = [f"passage : {text}" for text in texts]

    embeddings = model.encode(
    texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    )
    return embeddings
    

In [7]:
client = OpenSearch(
    hosts = [{"hosts": OPENSEARCH_HOST, "port": OPENSEARCH_PORT}],
    http_compress = True,
    timeout = 30,
    max_retries = 3,
    retry_on_timeout = True,
)
try:
    info = client.info()
    print(f"Successfully connect to OpenSearch {info['version']['number']}")
except Exception as e:
    print(f"Failed to connect to OpenSearch: {e}")
    print("Make sure OpenSearch is running on localhost:9200")
    raise


Successfully connect to OpenSearch 2.19.2


In [8]:
def create_index_config() -> Dict[str, Any]:
    """
    Creates the index configuration with mappings for text, embeddings, and metadata.
    
    Returns:
        Dict[str, Any]: The index configuration.
    """
    config = {
        "settings": {
            "index": {
                "number_of_shards": 1,
                "number_of_replicas": 0,
                "knn": True,
            }
        },
        "mappings":{
            "properties": {
                "text": {
                    "type": "text"
                },
                "embedding": {
                    "type": "knn_vector",
                    "dimension": EMBEDDING_DIMENSIONS,
                    "method": {
                        "engine":"faiss",
                        "space_type": "l2",
                        "name": "hnsw",
                        "parameters": {}
                    }
                },
                "document_name": {
                    "type": "keyword"
                }
            }
        }
    }
    return config

index_config = create_index_config()
print("\nIndex Configuration:")
print(json.dumps(index_config, indent=2))


Index Configuration:
{
  "settings": {
    "index": {
      "number_of_shards": 1,
      "number_of_replicas": 0,
      "knn": true
    }
  },
  "mappings": {
    "properties": {
      "text": {
        "type": "text"
      },
      "embedding": {
        "type": "knn_vector",
        "dimension": 384,
        "method": {
          "engine": "faiss",
          "space_type": "l2",
          "name": "hnsw",
          "parameters": {}
        }
      },
      "document_name": {
        "type": "keyword"
      }
    }
  }
}


In [9]:
if not client.indices.exists(index=OPENSEARCH_INDEX):
    response = client.indices.create(index=OPENSEARCH_INDEX, body=index_config)
    print(f"\nCreated index {OPENSEARCH_INDEX} with response: {response}")
else:
    print(f"\nIndex {OPENSEARCH_INDEX} already exists")


Index documents already exists


In [10]:
from opensearchpy.exceptions import NotFoundError
pipeline_name = "nlp-search-pipeline"

try:
    result = client.transport.perform_request(
        "GET",
        f"/_search/pipeline/{pipeline_name}"
    )
    print(f"\n Search pipeline {pipeline_name} exists.")
    print(result)
except NotFoundError:
    print(f"\n Search pipeline {pipeline_name} not exists.")
except Exception as e:
    print(f"\n error {e}")


 Search pipeline nlp-search-pipeline exists.
{'nlp-search-pipeline': {'description': 'Post processor for hybrid search', 'phase_results_processors': [{'normalization-processor': {'normalization': {'technique': 'min_max'}, 'combination': {'technique': 'arithmetic_mean', 'parameters': {'weights': [0.3, 0.7]}}}}]}}


In [12]:
from pathlib import Path

import pytesseract
from pdf2image import convert_from_path

pdf_path = "attention is all you need.pdf"

tesseract_path = Path(r"C:\Program Files\Tesseract-OCR\tesseract.exe")
if not tesseract_path.is_file():
    raise FileNotFoundError(f"Tesseract not found: {tesseract_path}")

pytesseract.pytesseract.tesseract_cmd = str(tesseract_path)

reader = PdfReader(pdf_path)

page_texts = []

for page_number in range(1, len(reader.pages) + 1):
    images = convert_from_path(
        pdf_path,
        dpi=300,
        first_page=page_number,
        last_page=page_number,
    )

    try:
        page_text = pytesseract.image_to_string(images[0], lang="eng")
        page_texts.append(page_text.strip())
    finally:
        for image in images:
            image.close()

    print(f"OCR completed: page {page_number}/{len(reader.pages)}")

text = "\n\n".join(page_texts)
cleaned_text = clean_text(text)

if not cleaned_text:
    raise ValueError("OCR did not recognize any text in the PDF.")

chunks = chunk_text(
    cleaned_text,
    chunk_size=TEXT_CHUNK_SIZE,
    overlap=TEXT_CHUNK_OVERLAP,
)

print(f"\nExtracted {len(cleaned_text)} characters")
print(f"Created {len(chunks)} chunks")
print("\nSample extracted text:")
print(cleaned_text[:500])

OCR completed: page 1/15
OCR completed: page 2/15
OCR completed: page 3/15
OCR completed: page 4/15
OCR completed: page 5/15
OCR completed: page 6/15
OCR completed: page 7/15
OCR completed: page 8/15
OCR completed: page 9/15
OCR completed: page 10/15
OCR completed: page 11/15
OCR completed: page 12/15
OCR completed: page 13/15
OCR completed: page 14/15
OCR completed: page 15/15

Extracted 39737 characters
Created 16 chunks

Sample extracted text:
1706.03762v7 [cs.CL] 2 Aug 2023
arXiv
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
Attention Is All You Need
Ashish Vaswani* Noam Shazeer* Niki Parmar* Jakob Uszkoreit* Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com
Llion Jones* Aidan N. Gomez* ' Lukasz Kaiser* Google Research University of 


In [13]:
pdf_file_name = pdf_path.replace(".pdf","")

embeddings = generate_embedding(chunks)
print(f"Generated {len(embeddings)} embeddings")
print(f"Embedding shape: {embeddings[0].shape}")
print(embeddings[0])

documents_to_index =[{
    "doc_id": f"{pdf_file_name}_{i}",
    "text": chunk,
    "embedding": embedding,
    "document_name": pdf_file_name,
    } for i, (chunk, embedding) in enumerate(zip(chunks, embeddings))
]
print(f"\nPrepared {len(documents_to_index)} documents for indexing")

Batches: 100%|██████████| 1/1 [00:01<00:00,  1.85s/it]

Generated 16 embeddings
Embedding shape: (384,)
[-1.35010257e-01 -1.41720846e-01  5.03768139e-02 -1.85307115e-02
  4.10707034e-02  6.66417778e-02 -7.60911554e-02 -1.88651197e-02
  1.12066381e-01 -2.75104083e-02  1.36285489e-02 -6.95466697e-02
 -6.87148869e-02  3.90506983e-02 -9.24987346e-02  9.42003075e-03
  8.76444206e-03  4.71563227e-02 -5.16732447e-02 -6.94512650e-02
 -1.23239933e-02  8.19378570e-02 -1.41413079e-03 -6.09339820e-03
 -2.67395619e-02 -2.96806451e-03 -2.53636446e-02 -2.30979528e-02
  1.65034905e-02 -1.16158575e-02  4.69618961e-02 -3.51247750e-02
 -1.97320785e-02  7.83743709e-02 -6.05980828e-02  6.00055568e-02
 -5.90721630e-02 -2.01446246e-02  2.96721067e-02 -3.41780521e-02
  3.96177657e-02 -8.63712654e-03 -2.76927631e-02  2.19846424e-02
  1.24715075e-01  6.64417297e-02 -4.14019544e-03 -1.73117463e-02
 -2.07056217e-02 -3.05130985e-02 -4.34733070e-02  8.49924311e-02
 -9.47340578e-03  1.21021844e-01 -9.17194132e-03 -1.99029762e-02
 -2.44567208e-02 -4.60780375e-02  1.403449

In [14]:
documents_to_index

[{'doc_id': 'attention is all you need_0',
  'text': "1706.03762v7 [cs.CL] 2 Aug 2023 arXiv Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani* Noam Shazeer* Niki Parmar* Jakob Uszkoreit* Google Brain Google Brain Google Research Google Research avaswani@google.com noam@google.com nikip@google.com usz@google.com Llion Jones* Aidan N. Gomez* ' Lukasz Kaiser* Google Research University of Toronto Google Brain llion@google.com aidan@cs.toronto.edu lukaszkaiser@google.com Illia Polosukhin* * illia.polosukhin@gmail.com Abstract The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, bas

In [15]:
actions = []

for doc in documents_to_index:
    if ASYMETRIC_EMBEDDING:
        prefixed_text = f"passage:{doc['text']}"
    else:
        prefixed_text = doc['text']
    action = {
        "_index": OPENSEARCH_INDEX,
        "_id": doc["doc_id"],
        "_source": {
            "text": prefixed_text,
            "embedding": doc["embedding"],
            "document_name": doc['document_name'],
        }
    }
    actions.append(action)

In [16]:
try:
    success, errors = helpers.bulk(client, actions, raise_on_error=True)
    if errors:
        print(f"Indexed {success} documents with {len(errors)} errors")
        print(f"First error: {errors[0]}")
    else:
        print(f"Successfully indexed {success} documents")
except Exception as e:
    print(f"Error during bulk loading {e}")

Successfully indexed 16 documents


In [17]:
query_text = "Model Architure"

query_embedding = generate_embedding([query_text])[0].tolist()

top_k = 3

query_body = {
    "_source": {"exclude": ["embedding"]},
    "query": {
        "hybrid": {
            "queries": [
                {"match": {"text": {"query": query_text}}},
                {
                    "knn": {
                        "embedding": {
                            "vector": query_embedding,
                            "k": top_k
                        }
                    }
                }
            ]
        }
    },
    "size": top_k
}

response = client.search(
        index=OPENSEARCH_INDEX, body=query_body, search_pipeline="nlp-search-pipeline"
    )

print(f"\nTop {top_k} results for query: '{query_text}'\n")
for i, hit in enumerate(response['hits']['hits'], 1):
    print(f"Result {i}:")
    print(json.dumps(hit['_source'], indent=2))
    print("-" * 60)

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.94it/s]



Top 3 results for query: 'Model Architure'

Result 1:
{
  "document_name": "attention is all you need",
  "text": "41.0, outperforming all of the previously published single models, at less than 1/4 the training cost of the previous state-of-the-art model. The Transformer (big) model trained for English-to-French used dropout rate Pg) = 0.1, instead of 0.3. For the base models, we used a single model obtained by averaging the last 5 checkpoints, which were written at 10-minute intervals. For the big models, we averaged the last 20 checkpoints. We used beam search with a beam size of 4 and length penalty a = 0.6 [38]. These hyperparameters were chosen after experimentation on the development set. We set the maximum output length during inference to input length + 50, but terminate early when possible [38]. Table 2 summarizes our results and compares our translation quality and training costs to other model architectures from the literature. We estimate the number of floating point oper